In [1]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import torch
import pyarrow as pa
import pyarrow.parquet as pq

# 1. Define paths
PROJECT_ROOT = r"C:\Users\aseem\OneDrive\Dokumen\aerospatium"
RAW_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

static_path = os.path.join(PROCESSED_DATA_DIR, "h3_static_features.parquet")
assert os.path.exists(static_path), f"Missing {static_path}. Run Notebook 01 from Version A first!"

df_static = pd.read_parquet(static_path)
N = len(df_static)
print(f"Loaded static H3 table: {N} cells.")

# 2. Construct Static Feature Matrix X_static [N, 11]
# Features:
# 0: Normalized Elevation (mean=0, std=1)
# 1: Normalized Slope
# 2: Aspect sin
# 3: Aspect cos
# 4: One-hot Grass (FM 1, 2)
# 5: One-hot Shrub (FM 4)
# 6: One-hot Timber (FM 10)
# 7: One-hot Non-burnable (FM 99)
# 8: Fuel frac grass
# 9: Fuel frac shrub
# 10: USDM Drought Category (normalized 0-4 -> 0-1)

elev_norm = ((df_static["elevation_mean_m"] - df_static["elevation_mean_m"].mean()) / df_static["elevation_mean_m"].std()).values
slope_norm = ((df_static["slope_mean_deg"] - df_static["slope_mean_deg"].mean()) / df_static["slope_mean_deg"].std()).values
aspect_sin = np.sin(df_static["aspect_mean_rad"].values)
aspect_cos = np.cos(df_static["aspect_mean_rad"].values)

is_grass = df_static["fuel_model"].isin([1, 2]).astype(np.float32).values
is_shrub = (df_static["fuel_model"] == 4).astype(np.float32).values
is_timber = (df_static["fuel_model"] == 10).astype(np.float32).values
is_nonburn = (df_static["fuel_model"] == 99).astype(np.float32).values

frac_grass = df_static["fuel_frac_grass"].values
frac_shrub = df_static["fuel_frac_shrub"].values
drought_norm = (df_static["drought_category"].values / 4.0).astype(np.float32)

X_static = np.stack([
    elev_norm, slope_norm, aspect_sin, aspect_cos,
    is_grass, is_shrub, is_timber, is_nonburn,
    frac_grass, frac_shrub, drought_norm
], axis=-1).astype(np.float32)

print(f"X_static shape: {X_static.shape} | dtypes: {X_static.dtype}")

# 3. Ingest Dynamic Weather Tensor across T=12 Horizons X_dyn [12, N, 4]
weather_nc_path = os.path.join(RAW_DATA_DIR, "weather", "era5_hourly_aoi.nc")
ds_weather = xr.open_dataset(weather_nc_path).sortby("latitude")

lats = df_static["centroid_lat"].values
lons = df_static["centroid_lon"].values
points_lat = xr.DataArray(lats, dims="points")
points_lon = xr.DataArray(lons, dims="points")

# Extract first 12 hours (00:00 to 11:00 UTC)
T = 12
u10_seq = ds_weather["u10"].isel(time=slice(0, T)).interp(latitude=points_lat, longitude=points_lon, method="linear").values # [12, N]
v10_seq = ds_weather["v10"].isel(time=slice(0, T)).interp(latitude=points_lat, longitude=points_lon, method="linear").values # [12, N]
t2m_seq = ds_weather["t2m"].isel(time=slice(0, T)).interp(latitude=points_lat, longitude=points_lon, method="linear").values # [12, N]
d2m_seq = ds_weather["d2m"].isel(time=slice(0, T)).interp(latitude=points_lat, longitude=points_lon, method="linear").values # [12, N]

# Compute derived RH and normalizations
tc_seq = t2m_seq - 273.15
td_seq = d2m_seq - 273.15
rh_seq = 100.0 * np.exp(17.625 * td_seq / (243.04 + td_seq)) / np.exp(17.625 * tc_seq / (243.04 + tc_seq))

# Normalize dynamic features: [U10, V10, Temp_norm, RH_norm]
temp_norm = (tc_seq - 25.0) / 15.0      # centered around typical fire weather
rh_norm = (rh_seq - 30.0) / 30.0

X_dyn = np.stack([u10_seq, v10_seq, temp_norm, rh_norm], axis=-1).astype(np.float32) # [12, N, 4]
print(f"X_dyn shape: {X_dyn.shape} | (T={T}, N={N}, Features=4)")

# 4. Save Processed Tensor Arrays as NumPy Binary Archives
tensor_output_path = os.path.join(PROCESSED_DATA_DIR, "gnn_feature_tensors.npz")
np.savez_compressed(
    tensor_output_path,
    X_static=X_static,
    X_dyn=X_dyn,
    h3_id=df_static["h3_id"].values
)

print(f"\nSaved GNN input tensors to: {tensor_output_path}")
print(f"Size on disk: {os.path.getsize(tensor_output_path) / 1024:.2f} KB")

Loaded static H3 table: 2526 cells.
X_static shape: (2526, 11) | dtypes: float32
X_dyn shape: (12, 2526, 4) | (T=12, N=2526, Features=4)

Saved GNN input tensors to: C:\Users\aseem\OneDrive\Dokumen\aerospatium\data\processed\gnn_feature_tensors.npz
Size on disk: 490.96 KB
